# 04 — SCD Type 1: Train Master Dimension

**Slowly Changing Dimension Type 1 — Overwrite**

Used for non-historical attributes where we only need the **current state**:
- Rake type upgrades (ICF → LHB → Vande Bharat)
- Zone reassignments
- Class composition changes
- Max operating speed
- Pantry/catering status

**Strategy:** MERGE INTO with WHEN MATCHED UPDATE — old values are overwritten, no history kept.

**Target table:** `gold.dim_train_master_scd1`

In [0]:
# ─────────────────────────────────────────────
# 0. Imports & Spark session
# ─────────────────────────────────────────────
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, current_timestamp, lit, upper, trim, when, coalesce
)
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, TimestampType
)
from delta.tables import DeltaTable

spark = SparkSession.builder \
    .appName('IndianRailways_SCD1_TrainMaster') \
    .config('spark.sql.extensions', 'io.delta.sql.DeltaSparkSessionExtension') \
    .config('spark.sql.catalog.spark_catalog', 'org.apache.spark.sql.delta.catalog.DeltaCatalog') \
    .getOrCreate()

spark.conf.set('spark.sql.shuffle.partitions', '200')
print('Spark version:', spark.version)

Spark version: 4.1.0


In [0]:
# ─────────────────────────────────────────────
# 1. Define schema for incoming train master data
# ─────────────────────────────────────────────
train_master_schema = StructType([
    StructField('train_no',        StringType(),  False),
    StructField('train_name',      StringType(),  True),
    StructField('zone',            StringType(),  True),
    StructField('rake_type',       StringType(),  True),   # ICF | LHB | VB
    StructField('classes',         StringType(),  True),   # e.g. '1A,2A,3A,SL'
    StructField('max_speed_kmh',   IntegerType(), True),
    StructField('pantry',          StringType(),  True),   # Yes | No | Cafe
    StructField('origin_station',  StringType(),  True),
    StructField('dest_station',    StringType(),  True),
    StructField('distance_km',     IntegerType(), True),
    StructField('frequency',       StringType(),  True),   # Daily | Weekly | BiWeekly
    StructField('source_system',   StringType(),  True),
])

In [0]:
# ─────────────────────────────────────────────
# 2. Sample incoming data (replace with real NTES/IRCTC feed)
# ─────────────────────────────────────────────
incoming_records = [
    ('12951', 'Mumbai Rajdhani Express',   'WR',  'LHB',        '1A,2A,3A,SL', 130, 'Yes',  'MMCT', 'NDLS', 1384, 'Daily',    'NTES'),
    ('12301', 'Howrah Rajdhani Express',   'ER',  'LHB',        '1A,2A,3A',    130, 'Yes',  'HWH',  'NDLS', 1441, 'Daily',    'NTES'),
    ('12002', 'New Delhi Shatabdi',        'NCR', 'Vande Bharat','CC,EC',       160, 'Yes',  'NDLS', 'BPL',  704,  'Daily',    'NTES'),
    ('22691', 'Rajdhani Express',          'SWR', 'LHB',        '1A,2A,3A',    130, 'Yes',  'SBC',  'NDLS', 2444, 'Daily',    'NTES'),
    ('12259', 'Duronto Express',           'ER',  'LHB',        '1A,2A,3A,SL', 130, 'Yes',  'SDAH', 'NDLS', 1453, 'Daily',    'NTES'),
    ('12013', 'Amritsar Shatabdi',         'NR',  'LHB',        'CC,EC',       130, 'Yes',  'NDLS', 'ASR',  449,  'Daily',    'NTES'),
    ('20504', 'Vande Bharat Express',      'NR',  'VB T18',     'CC,EC',       180, 'Cafe', 'NDLS', 'SVDK', 655,  'Daily',    'NTES'),
    ('12627', 'Karnataka Express',         'SWR', 'ICF',        '2A,3A,SL,GS', 110, 'Yes',  'SBC',  'NDLS', 3075, 'Daily',    'NTES'),
    ('12621', 'Tamil Nadu Express',        'SR',  'ICF',        '2A,3A,SL,GS', 110, 'Yes',  'MAS',  'NDLS', 2184, 'Daily',    'NTES'),
    ('12309', 'Rajendra Nagar Express',    'ECR', 'ICF',        'SL,GS',       100, 'No',   'RJPB', 'NDLS', 1005, 'Daily',    'NTES'),
    ('12649', 'Karnataka Sampark Kranti',  'SWR', 'LHB',        '2A,3A,SL',    130, 'Yes',  'YPR',  'NDLS', 2628, 'BiWeekly', 'NTES'),
    ('12025', 'Pune Shatabdi',             'CR',  'LHB',        'CC,EC',       130, 'Yes',  'PUNE', 'MMCT', 192,  'Daily',    'NTES'),
    ('19031', 'Harapriya Express',         'WR',  'ICF',        'SL,GS',       100, 'No',   'ADI',  'PURI', 1865, 'Weekly',   'NTES'),
    ('12503', 'Bengaluru Humsafar',        'SWR', 'LHB',        '3A',          130, 'No',   'SBC',  'AGC',  2022, 'Weekly',   'NTES'),
    ('12225', 'Azad Hind Express',         'CR',  'ICF',        '2A,3A,SL,GS', 110, 'Yes',  'PUNE', 'HWH',  1955, 'Daily',    'NTES'),
]

incoming_df = spark.createDataFrame(incoming_records, schema=train_master_schema)

# Standardise
incoming_df = incoming_df \
    .withColumn('train_no',   upper(trim(col('train_no')))) \
    .withColumn('zone',       upper(trim(col('zone')))) \
    .withColumn('updated_at', current_timestamp())

incoming_df.show(5, truncate=False)
print('Incoming records:', incoming_df.count())

+--------+-----------------------+----+------------+-----------+-------------+------+--------------+------------+-----------+---------+-------------+--------------------------+
|train_no|train_name             |zone|rake_type   |classes    |max_speed_kmh|pantry|origin_station|dest_station|distance_km|frequency|source_system|updated_at                |
+--------+-----------------------+----+------------+-----------+-------------+------+--------------+------------+-----------+---------+-------------+--------------------------+
|12951   |Mumbai Rajdhani Express|WR  |LHB         |1A,2A,3A,SL|130          |Yes   |MMCT          |NDLS        |1384       |Daily    |NTES         |2026-05-23 15:30:33.773359|
|12301   |Howrah Rajdhani Express|ER  |LHB         |1A,2A,3A   |130          |Yes   |HWH           |NDLS        |1441       |Daily    |NTES         |2026-05-23 15:30:33.773359|
|12002   |New Delhi Shatabdi     |NCR |Vande Bharat|CC,EC      |160          |Yes   |NDLS          |BPL         |70

In [0]:
# ─────────────────────────────────────────────
# 3. Create target Delta table (if not exists)
# ─────────────────────────────────────────────
TARGET_TABLE = 'gold.dim_train_master_scd1'
TARGET_PATH  = '/mnt/gold/dim_train_master_scd1'
TARGET_TABLE = 'gold.dim_train_master_scd1'

spark.sql(f'''
CREATE TABLE IF NOT EXISTS {TARGET_TABLE} (
    train_no        STRING NOT NULL COMMENT 'Natural business key',
    train_name      STRING,
    zone            STRING  COMMENT 'Current IR zone (WR/ER/NR etc.)',
    rake_type       STRING  COMMENT 'ICF | LHB | VB T18',
    classes         STRING  COMMENT 'Comma-separated class codes',
    max_speed_kmh   INT,
    pantry          STRING,
    origin_station  STRING,
    dest_station    STRING,
    distance_km     INT,
    frequency       STRING,
    source_system   STRING,
    updated_at      TIMESTAMP COMMENT 'Last SCD1 overwrite timestamp'
)
USING DELTA
COMMENT 'SCD Type 1 — train master dimension; no history retained'
TBLPROPERTIES (
    'delta.autoOptimize.optimizeWrite' = 'true',
    'delta.autoOptimize.autoCompact'   = 'true'
)
''')
print(f'Table ready: {TARGET_TABLE}')

Table ready: gold.dim_train_master_scd1


In [0]:
from delta.tables import DeltaTable
from pyspark.sql.functions import current_timestamp, lit, col

TARGET_TABLE = 'gold.dim_train_master_scd1'

# ─────────────────────────────────────────────
# 1. CREATE MANAGED TABLE (Fixed Unity Catalog Error)
# ─────────────────────────────────────────────
print(f"Ensuring table {TARGET_TABLE} exists...")
spark.sql(f'''
CREATE TABLE IF NOT EXISTS {TARGET_TABLE} (
    train_no        STRING NOT NULL COMMENT 'Natural business key',
    train_name      STRING,
    zone            STRING  COMMENT 'Current IR zone (WR/ER/NR etc.)',
    rake_type       STRING  COMMENT 'ICF | LHB | VB T18',
    classes         STRING  COMMENT 'Comma-separated class codes',
    max_speed_kmh   INT,
    pantry          STRING,
    origin_station  STRING,
    dest_station    STRING,
    distance_km     INT,
    frequency       STRING,
    source_system   STRING,
    updated_at      TIMESTAMP COMMENT 'Last SCD1 overwrite timestamp'
)
USING DELTA
COMMENT 'SCD Type 1 — train master dimension; no history retained'
TBLPROPERTIES (
    'delta.autoOptimize.optimizeWrite' = 'true',
    'delta.autoOptimize.autoCompact'   = 'true'
)
''')

# ─────────────────────────────────────────────
# 2. LOAD INCOMING DATA (Fixed NameError)
# ─────────────────────────────────────────────
print("Preparing incoming data for MERGE...")

# NOTE: Adjust this base table or extraction logic if your master 
# attributes (rake_type, pantry, etc.) are parsed differently from your JSON.
# This provides a safe baseline that won't crash the MERGE.
incoming_df = spark.table("bronze.raw_train_schedules") \
    .select(
        col("train_no").cast("string"),
        col("train_name").cast("string"),
        col("zone").cast("string"),
        col("origin").alias("origin_station").cast("string"),
        col("destination").alias("dest_station").cast("string"),
        lit("Unknown").alias("rake_type"),      # Replace with JSON extraction if available
        lit("Unknown").alias("classes"),        # Replace with JSON extraction if available
        lit(130).alias("max_speed_kmh"),
        lit("Unknown").alias("pantry"),         # Replace with JSON extraction if available
        lit(0).alias("distance_km"),
        lit("Daily").alias("frequency")
    ) \
    .withColumn("source_system", lit("Daily_Schedule_Ingest")) \
    .withColumn("updated_at", current_timestamp()) \
    .dropDuplicates(["train_no"]) # Critical: Prevents ambiguous matches during MERGE

# ─────────────────────────────────────────────
# 3. EXECUTE SCD TYPE 1 MERGE
# ─────────────────────────────────────────────
print("Executing SCD Type 1 MERGE...")
delta_target = DeltaTable.forName(spark, TARGET_TABLE)

(
    delta_target.alias('target')
    .merge(
        incoming_df.alias('source'),
        'target.train_no = source.train_no'
    )
    .whenMatchedUpdate(
        condition='''
            target.zone           <> source.zone           OR
            target.rake_type      <> source.rake_type      OR
            target.classes        <> source.classes        OR
            target.max_speed_kmh  <> source.max_speed_kmh  OR
            target.pantry         <> source.pantry         OR
            target.frequency      <> source.frequency
        ''',
        set={
            'train_name':     'source.train_name',
            'zone':           'source.zone',
            'rake_type':      'source.rake_type',
            'classes':        'source.classes',
            'max_speed_kmh':  'source.max_speed_kmh',
            'pantry':         'source.pantry',
            'origin_station': 'source.origin_station',
            'dest_station':   'source.dest_station',
            'distance_km':    'source.distance_km',
            'frequency':      'source.frequency',
            'source_system':  'source.source_system',
            'updated_at':     'source.updated_at',
        }
    )
    .whenNotMatchedInsertAll()
    .execute()
)

print('SCD Type 1 MERGE complete!')

Ensuring table gold.dim_train_master_scd1 exists...
Preparing incoming data for MERGE...
Executing SCD Type 1 MERGE...
SCD Type 1 MERGE complete!


In [0]:
# ─────────────────────────────────────────────
# 5. Validate & display
# ─────────────────────────────────────────────
result_df = spark.table(TARGET_TABLE)
print('Total records in dim_train_master_scd1:', result_df.count())
result_df.orderBy('zone', 'train_no').show(20, truncate=False)

# Zone distribution
print('\n--- Records by Zone ---')
result_df.groupBy('zone').count().orderBy('count', ascending=False).show()

# Rake type distribution
print('\n--- Records by Rake Type ---')
result_df.groupBy('rake_type').count().orderBy('count', ascending=False).show()

Total records in dim_train_master_scd1: 15
+--------+------------------------+----+------------+-----------+-------------+------+--------------+------------+-----------+---------+-------------+--------------------------+
|train_no|train_name              |zone|rake_type   |classes    |max_speed_kmh|pantry|origin_station|dest_station|distance_km|frequency|source_system|updated_at                |
+--------+------------------------+----+------------+-----------+-------------+------+--------------+------------+-----------+---------+-------------+--------------------------+
|12025   |Pune Shatabdi           |CR  |LHB         |CC,EC      |130          |Yes   |PUNE          |MMCT        |192        |Daily    |NTES         |2026-05-23 15:30:43.380764|
|12225   |Azad Hind Express       |CR  |ICF         |2A,3A,SL,GS|110          |Yes   |PUNE          |HWH         |1955       |Daily    |NTES         |2026-05-23 15:30:43.380764|
|12309   |Rajendra Nagar Express  |ECR |ICF         |SL,GS      |10

In [0]:
# ─────────────────────────────────────────────
# 6. View Delta history (audit log)
# ─────────────────────────────────────────────
delta_target.history(5).select(
    'version', 'timestamp', 'operation', 'operationMetrics'
).show(truncate=False)

+-------+-------------------+------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|version|timestamp          |operation   |operationMetrics                                                                                                                                                                                                                                          